# Mystery Corpus Experiments — R-GCN on Detective Relational Graph

This notebook handles all mystery-corpus-specific logic:
- Dataset loading via `load_mystery_graphs_updated.py`
- Model training (or checkpoint loading)
- Link prediction evaluation
- Interactive character/relation prediction

The R-GCN model, training loop, and evaluation utilities live in `rgcn_model.py`.
The data loader lives in `load_mystery_graphs_updated.py`.
Make sure both files are in the same directory as this notebook.

---
## 0. Imports

In [1]:
%pip install torch torch-geometric
%pip install "numpy==1.26.4" --force-reinstall


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/95/12/8f2020a8e8b8383ac0177dc9570aad031a3beb12e38847f7129bacd96228/numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata
  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl (20.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
py-tgb 2.2.0 requires numpy<3.0.0,>=2.0.2, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip

In [2]:
import os, pickle
import numpy as np
import torch
import matplotlib.pyplot as plt

# ── Import model library ──────────────────────────────────────────────────────
from rgcn_model import (
    DEVICE,
    RelationalGraph,
    RGCNLinkPredictor,
    RGCNLinkPredictorPyG,
    PYG_AVAILABLE,
    train,
    evaluate,
    plot_history,
)

# ── Import mystery data loader ────────────────────────────────────────────────
from load_mystery_graphs import load_mystery_graphs

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"PyG     : {'available' if PYG_AVAILABLE else 'not installed'}")

/Users/Isaac/290-Final-Project/R-GCN/rgcn_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch : 2.2.2
Device  : cpu
PyG     : available


In [3]:
# ── Project folders ───────────────────────────────────────────────────────────
# Edit PROJECT_DIR if your JSON files live somewhere other than ./json_dir
PROJECT_DIR    = os.getcwd()
JSON_DIR       = os.path.join(PROJECT_DIR, "json_dir")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"JSON directory       : {JSON_DIR}")
print(f"Checkpoint directory : {CHECKPOINT_DIR}")

JSON directory       : /Users/Isaac/290-Final-Project/R-GCN/json_dir
Checkpoint directory : /Users/Isaac/290-Final-Project/R-GCN/checkpoints


---
## 1. Mystery Corpus Dataset Loading

Loads all mystery JSON files from `JSON_DIR` and merges them into a single
`RelationalGraph` with 8 coarse relation types (v3 schema).

The graph is pickled after the first load so subsequent runs skip JSON parsing.

In [4]:
# ── Checkpoint paths ──────────────────────────────────────────────────────────
SCRATCH_CKPT = os.path.join(CHECKPOINT_DIR, "rgcn_scratch_detective.pt")
PYG_CKPT     = os.path.join(CHECKPOINT_DIR, "rgcn_pyg_detective.pt")
GRAPH_CKPT   = os.path.join(CHECKPOINT_DIR, "detective_graph.pkl")

# Load graph from checkpoint if available, otherwise parse JSON corpus
if os.path.exists(GRAPH_CKPT):
    print(f"Loading graph from checkpoint: {GRAPH_CKPT}")
    with open(GRAPH_CKPT, "rb") as f:
        det_graph = pickle.load(f)
    print("Graph loaded from checkpoint — skipping JSON parsing.")
else:
    print("No graph checkpoint found — loading from JSON corpus...")
    det_graph = load_mystery_graphs(JSON_DIR)
    with open(GRAPH_CKPT, "wb") as f:
        pickle.dump(det_graph, f)
    print(f"Graph saved to {GRAPH_CKPT}")

det_graph.summary()

No graph checkpoint found — loading from JSON corpus...
Story split — train: 462, val: 57, test: 57
Nodes          : 14,020
Relation types : 8  (coarsened from 536 intermediate forms, residual → social)
Train edges    : 12,866
Val edges      : 1,583
Test edges     : 1,498
Node feat dim  : 10

Coarse relation distribution:
  0  kills                  779  (  4.9%)
  1  harms                  619  (  3.9%)
  2  investigates         1,900  ( 11.9%)
  3  deceives             1,153  (  7.2%)
  4  personal_bond        3,070  ( 19.3%)
  5  professional         3,856  ( 24.2%)
  6  spatial              2,925  ( 18.3%)
  7  social               1,645  ( 10.3%)
Graph saved to /Users/Isaac/290-Final-Project/R-GCN/checkpoints/detective_graph.pkl
RelationalGraph: mystery-corpus
  Nodes         : 14,020
  Relation types: 8
  Train edges   : 12,866
  Val edges     : 1,583
  Test edges    : 1,498
  Node features : yes


In [5]:
# Verify label samples
print("Sample node labels:")
print(list(det_graph.node_labels.items())[:5])
print("\nRelation labels:")
for idx, label in sorted(det_graph.rel_labels.items()):
    count = (det_graph.train_edges[:, 2] == idx).sum().item()
    pct   = 100 * count / len(det_graph.train_edges)
    print(f"  {idx}: {label:<20} {count:>6,} train edges  ({pct:4.1f}%)")

Sample node labels:
[(0, 'Iris Henderson'), (1, 'Miss Froy'), (2, 'Gilbert Redman'), (3, 'Charters and Caldicott'), (4, 'Eric Todhunter')]

Relation labels:
  0: kills                   628 train edges  ( 4.9%)
  1: harms                   498 train edges  ( 3.9%)
  2: investigates          1,529 train edges  (11.9%)
  3: deceives                948 train edges  ( 7.4%)
  4: personal_bond         2,501 train edges  (19.4%)
  5: professional          3,120 train edges  (24.2%)
  6: spatial               2,313 train edges  (18.0%)
  7: social                1,329 train edges  (10.3%)


---
## 2. From-Scratch Model: Train or Load

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# node_features supplies the pre-computed 10-dim feature vectors from the JSON,
# so feat_dim=10 replaces the learned embedding table used for Wikidata.
det_model_scratch = RGCNLinkPredictor(
    num_nodes     = det_graph.num_nodes,
    num_relations = det_graph.num_relations,
    hidden_dim    = 64,
    num_layers    = 2,
    num_bases     = 30,
    feat_dim      = det_graph.node_features.shape[1],  # 10
    dropout       = 0.1,
)
print(f"Scratch model parameters: {sum(p.numel() for p in det_model_scratch.parameters()):,}")

if os.path.exists(SCRATCH_CKPT):
    print(f"Checkpoint found — loading scratch model from {SCRATCH_CKPT}")
    det_model_scratch.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
    det_model_scratch.to(DEVICE).eval()
    det_history_scratch = None
    print("Scratch model ready (training skipped).")
else:
    print("No checkpoint found — training scratch model...")
    det_history_scratch = train(
        det_model_scratch, det_graph,
        num_epochs=50, lr=1e-3, batch_size=4096, eval_every=5, neg_ratio=3,
    )
    torch.save(det_model_scratch.state_dict(), SCRATCH_CKPT)
    print(f"Scratch model saved to {SCRATCH_CKPT}")

In [ ]:
if det_history_scratch is not None:
    plot_history(det_history_scratch, "Mystery Corpus — R-GCN from scratch")
else:
    print("Model was loaded from checkpoint — no training history to plot.")

---
## 3. From-Scratch Model: Test Evaluation

In [ ]:
train_e    = det_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

test_metrics = evaluate(
    det_model_scratch, det_graph,
    det_graph.test_edges, edge_index, edge_type,
)
print("Test results (from-scratch model):")
for k, v in test_metrics.items():
    print(f"  {k:10s}: {v:.4f}")

---
## 4. PyG Model: Train or Load

In [ ]:
TRAIN_PYG = False  # set True to train the PyG variant

if PYG_AVAILABLE and TRAIN_PYG:
    det_model_pyg = RGCNLinkPredictorPyG(
        num_nodes     = det_graph.num_nodes,
        num_relations = det_graph.num_relations,
        hidden_dim    = 64,
        num_layers    = 2,
        num_bases     = 30,
        feat_dim      = det_graph.node_features.shape[1],  # 10
        dropout       = 0.1,
    )
    print(f"PyG model parameters: {sum(p.numel() for p in det_model_pyg.parameters()):,}")

    if os.path.exists(PYG_CKPT):
        print(f"Checkpoint found — loading PyG model from {PYG_CKPT}")
        det_model_pyg.load_state_dict(torch.load(PYG_CKPT, map_location=DEVICE))
        det_model_pyg.to(DEVICE).eval()
        det_history_pyg = None
        print("PyG model ready (training skipped).")
    else:
        print("No PyG checkpoint found — training...")
        det_history_pyg = train(
            det_model_pyg, det_graph,
            num_epochs=50, lr=1e-3, batch_size=4096, eval_every=5, neg_ratio=3,
        )
        torch.save(det_model_pyg.state_dict(), PYG_CKPT)
        print(f"PyG model saved to {PYG_CKPT}")
else:
    print("PyG training skipped (TRAIN_PYG=False or PyG not installed). "
          "Set TRAIN_PYG=True to enable.")
    det_model_pyg   = None
    det_history_pyg = None

In [ ]:
if PYG_AVAILABLE and TRAIN_PYG:
    if det_history_pyg is not None:
        plot_history(det_history_pyg, "Mystery Corpus — R-GCN (PyG)")
    else:
        print("PyG model was loaded from checkpoint — no training history to plot.")

---
## 5. Scratch vs PyG Comparison

In [ ]:
if PYG_AVAILABLE and det_history_scratch is not None and det_history_pyg is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Mystery Corpus: Scratch vs PyG", fontsize=13)
    for hist, label, ls in [
        (det_history_scratch, "Scratch", "-"),
        (det_history_pyg,     "PyG",     "--"),
    ]:
        axes[0].plot(hist["train_loss"], ls=ls, label=label)
        axes[1].plot(hist["val_mrr"],    ls=ls, label=f"{label} MRR")
        axes[1].plot(hist["val_hits10"], ls=ls, alpha=0.6, label=f"{label} Hits@10")
    axes[0].set_title("Loss"); axes[0].legend()
    axes[1].set_title("Val Metrics"); axes[1].legend()
    plt.tight_layout(); plt.show()
else:
    print("Comparison plot skipped — need both models trained in the same session.")

---
## 6. Link Prediction Inference

Pre-compute node embeddings once, then use them for `predict_object` and
`predict_relation`.

Node names in this graph are character/location/occupation/organization names
exactly as they appear in the JSON files (e.g. `"Blythe Connor"`, `"House"`,
`"Doctor"`).

In [ ]:
# ── Pre-compute node embeddings ───────────────────────────────────────────────
# Swap det_model_pyg for det_model_scratch here if you prefer.
train_e    = det_graph.train_edges.to(DEVICE)
edge_index = train_e[:, :2].t().contiguous()
edge_type  = train_e[:, 2]

active_model = det_model_scratch

with torch.no_grad():
    node_emb = active_model.encoder(
        edge_index, edge_type,
        node_features=det_graph.node_features.to(DEVICE),
        num_nodes=det_graph.num_nodes,
    )

model = active_model
print(f"Node embeddings shape: {node_emb.shape}")

In [ ]:
def predict_object(subject: str, relation: str, top_k: int = 10):
    """
    Given a subject node name and a coarse relation type (as strings),
    return the top_k most likely object nodes.

    Subject names are character/location/occupation/organization names
    exactly as stored in det_graph.node_labels.
    Relation names must match a key in det_graph.rel_labels, e.g.:
        'kills', 'harms', 'investigates', 'deceives',
        'personal_bond', 'professional', 'spatial', 'social'
    """
    node2int = {v: k for k, v in det_graph.node_labels.items()}
    rel2int  = {v: k for k, v in det_graph.rel_labels.items()}

    if subject not in node2int:
        print(f"Subject '{subject}' not found in node labels.")
        matches = [n for n in node2int if subject.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        else:
            print("  No partial matches found.")
        return

    if relation not in rel2int:
        print(f"Relation '{relation}' not found.")
        print(f"  Valid relations: {list(rel2int.keys())}")
        return

    s_idx = node2int[subject]
    r_idx = rel2int[relation]

    with torch.no_grad():
        h_s = node_emb[s_idx].unsqueeze(0)
        r   = model.decoder.relation_emb(
                  torch.tensor([r_idx], device=DEVICE))
        scores = (h_s * r * node_emb).sum(dim=-1)

    top_scores, top_indices = scores.topk(top_k)

    print(f"\n  '{subject}'  --[{relation}]-->  ???\n")
    print(f"  {'Rank':<6} {'Node':<35} {'Score':>8}")
    print("  " + "-" * 52)
    for rank, (idx, score) in enumerate(
            zip(top_indices.tolist(), top_scores.tolist()), 1):
        label = det_graph.node_labels.get(idx, str(idx))
        print(f"  {rank:<6} {label:<35} {score:>8.4f}")


def predict_relation(subject: str, obj: str, top_k: int = 5):
    """
    Given two node names, return the most likely coarse relation types between them.
    """
    node2int = {v: k for k, v in det_graph.node_labels.items()}

    if subject not in node2int:
        print(f"Subject '{subject}' not found.")
        matches = [n for n in node2int if subject.lower() in n.lower()]
        if matches:
            print(f"  Partial matches:", ", ".join(matches[:5]))
        return

    if obj not in node2int:
        print(f"Object '{obj}' not found.")
        matches = [n for n in node2int if obj.lower() in n.lower()]
        if matches:
            print(f"  Partial matches:", ", ".join(matches[:5]))
        return

    s_idx = node2int[subject]
    o_idx = node2int[obj]

    with torch.no_grad():
        h_s   = node_emb[s_idx]
        h_o   = node_emb[o_idx]
        all_r = model.decoder.relation_emb.weight  # (num_relations, hidden_dim)
        scores = (h_s.unsqueeze(0) * all_r * h_o.unsqueeze(0)).sum(dim=-1)

    top_scores, top_indices = scores.topk(min(top_k, det_graph.num_relations))

    print(f"\n  '{subject}'  --[???]-->  '{obj}'\n")
    print(f"  {'Rank':<6} {'Relation':<25} {'Score':>8}")
    print("  " + "-" * 42)
    for rank, (idx, score) in enumerate(
            zip(top_indices.tolist(), top_scores.tolist()), 1):
        label = det_graph.rel_labels.get(idx, str(idx))
        print(f"  {rank:<6} {label:<25} {score:>8.4f}")

---
## 7. Example Predictions

In [ ]:
# Who does a character with a 'kills' relation most likely target?
# (Replace with an actual character name from your corpus.)
predict_object("Blythe Connor", "kills")

# What relation is most likely between two characters in the same story?
predict_relation("Blythe Connor", "Sam")

# What nodes is a location most spatially connected to?
predict_object("House", "spatial")

# What is a character most likely professionally linked to?
predict_object("Violet", "professional")

---
## 8. Graph Exploration Utilities

In [ ]:
def entity_stats(name: str):
    """Show how many training edges a node appears in, broken down by relation type."""
    node2int = {v: k for k, v in det_graph.node_labels.items()}
    if name not in node2int:
        print(f"Node '{name}' not found.")
        matches = [n for n in node2int if name.lower() in n.lower()]
        if matches:
            print(f"  Partial matches ({len(matches)} found, showing first 5):",
                  ", ".join(matches[:5]))
        return

    idx        = node2int[name]
    as_subject = det_graph.train_edges[det_graph.train_edges[:, 0] == idx]
    as_object  = det_graph.train_edges[det_graph.train_edges[:, 1] == idx]
    print(f"'{name}' — {len(as_subject)} outgoing, {len(as_object)} incoming training edges")

    if len(as_subject):
        print("  Outgoing by relation:")
        for r_idx in range(det_graph.num_relations):
            c = (as_subject[:, 2] == r_idx).sum().item()
            if c:
                print(f"    {det_graph.rel_labels[r_idx]:<20} {c}")

    if len(as_object):
        print("  Incoming by relation:")
        for r_idx in range(det_graph.num_relations):
            c = (as_object[:, 2] == r_idx).sum().item()
            if c:
                print(f"    {det_graph.rel_labels[r_idx]:<20} {c}")


# Example — use a character name from your corpus
entity_stats("Blythe Connor")

In [ ]:
# Full relation distribution across train / val / test splits
print(f"{'Relation':<20} {'Train':>8} {'Val':>6} {'Test':>6}  {'Train %':>8}")
print("-" * 52)
for r_idx in range(det_graph.num_relations):
    label      = det_graph.rel_labels[r_idx]
    tr = (det_graph.train_edges[:, 2] == r_idx).sum().item()
    va = (det_graph.val_edges[:,   2] == r_idx).sum().item()
    te = (det_graph.test_edges[:,  2] == r_idx).sum().item()
    pct = 100 * tr / len(det_graph.train_edges)
    print(f"  {label:<18} {tr:>8,} {va:>6,} {te:>6,}  {pct:>7.1f}%")

In [ ]:
# All valid relation type names (use these strings in predict_object / predict_relation)
print("Valid relation types:")
for idx, label in sorted(det_graph.rel_labels.items()):
    print(f"  {idx}: {label}")

---
## 9. Manual Checkpoint Management

Cells below are commented out — uncomment only if you need to force-save or force-reload.

In [ ]:
# Force save
# torch.save(det_model_scratch.state_dict(), SCRATCH_CKPT)
# print(f"Scratch model saved to {SCRATCH_CKPT}")
#
# if PYG_AVAILABLE and det_model_pyg is not None:
#     torch.save(det_model_pyg.state_dict(), PYG_CKPT)
#     print(f"PyG model saved to {PYG_CKPT}")

In [ ]:
# Force reload scratch model
# det_model_scratch.load_state_dict(torch.load(SCRATCH_CKPT, map_location=DEVICE))
# det_model_scratch.to(DEVICE).eval()
# print("Scratch model reloaded.")

In [ ]:
# Force rebuild graph from JSON (e.g. after updating load_mystery_graphs_updated.py)
# import os
# if os.path.exists(GRAPH_CKPT):
#     os.remove(GRAPH_CKPT)
#     print(f"Deleted {GRAPH_CKPT} — re-run Section 1 to rebuild.")